# 📊 Notebook 01 — Exploratory Data Analysis & Cleaning
**Retail Sales Neural Network Project**

This notebook performs interactive EDA on the Superstore dataset:
- Data quality audit
- Sales trends over time
- Feature correlation heatmap
- Discount impact analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import load_raw_data, clean_data, engineer_datetime_features, engineer_business_features

# Plot style
plt.style.use('dark_background')
PALETTE = ['#00d4ff', '#ff6b6b', '#ffa500', '#10b981', '#7c5cbf']
sns.set_palette(PALETTE)

print('Libraries loaded ✓')

## 1. Load & Audit Raw Data

In [ ]:
df_raw = load_raw_data('../data/raw/sales.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Data quality report
quality = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'null_count': df_raw.isnull().sum(),
    'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'unique': df_raw.nunique()
})
quality[quality['null_count'] > 0].style.background_gradient(cmap='Reds', subset=['null_pct'])

In [ ]:
df = clean_data(df_raw)
df = engineer_datetime_features(df)
df = engineer_business_features(df)
print(f'After cleaning: {df.shape}')
df[['Sales','Profit','Discount','Quantity']].describe().round(2)

## 2. Sales Trends Over Time

In [ ]:
monthly = df.groupby(['year', 'month'])['Sales'].sum().reset_index()
monthly['period'] = pd.to_datetime(monthly.assign(day=1)[['year','month','day']])
monthly = monthly.sort_values('period')

fig, axes = plt.subplots(2, 1, figsize=(15, 10), facecolor='#0f1117')

# Monthly total sales
ax = axes[0]
ax.set_facecolor('#1a1d27')
ax.plot(monthly['period'], monthly['Sales'], color='#00d4ff', linewidth=2.5)
ax.fill_between(monthly['period'], monthly['Sales'], alpha=0.15, color='#00d4ff')
ax.set_title('Monthly Total Revenue', fontsize=14, fontweight='bold', color='white', pad=12)
ax.set_ylabel('Total Sales ($)', color='#94a3b8')
ax.tick_params(colors='#94a3b8')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
for spine in ax.spines.values(): spine.set_edgecolor('#333')

# Sales by category over time
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
colors = {'Furniture':'#ff6b6b', 'Technology':'#00d4ff', 'Office Supplies':'#ffa500'}
for cat, color in colors.items():
    cat_data = df[df['Category']==cat].groupby(['year','month'])['Sales'].sum().reset_index()
    cat_data['period'] = pd.to_datetime(cat_data.assign(day=1)[['year','month','day']])
    ax2.plot(cat_data['period'], cat_data['Sales'], label=cat, color=color, linewidth=2)
ax2.set_title('Sales by Category Over Time', fontsize=14, fontweight='bold', color='white', pad=12)
ax2.set_ylabel('Sales ($)', color='#94a3b8')
ax2.tick_params(colors='#94a3b8')
ax2.legend(facecolor='#1a1d27', labelcolor='white')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
for spine in ax2.spines.values(): spine.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('../data/processed/eda_sales_trends.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Chart saved ✓')

## 3. Feature Correlation Matrix

In [ ]:
num_cols = ['Sales','Profit','Quantity','Discount','discount_percentage','promo_active','month','year','day_of_week','is_weekend']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9), facecolor='#0f1117')
ax.set_facecolor('#0f1117')

mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
    center=0, linewidths=0.5, linecolor='#0f1117',
    annot_kws={'size': 9, 'color': 'white'},
    ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', color='white', pad=16)
ax.tick_params(colors='#94a3b8', labelsize=9)

plt.tight_layout()
plt.savefig('../data/processed/eda_correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 4. Discount Impact on Sales & Profit

In [ ]:
# Bin discounts for boxplot
df['discount_bin'] = pd.cut(df['Discount'], bins=[-0.01,0,0.1,0.2,0.3,0.5,1.0],
                             labels=['0%','1-10%','11-20%','21-30%','31-50%','51%+'])

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#0f1117')
fig.suptitle('Discount Impact Analysis', fontsize=15, fontweight='bold', color='white', y=1.02)

for ax in axes:
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values(): spine.set_edgecolor('#333')

# Boxplot: Discount bin vs Sales
sns.boxplot(data=df, x='discount_bin', y='Sales', ax=axes[0],
            palette=PALETTE, showfliers=False)
axes[0].set_title('Sales Distribution by Discount Level', color='white', fontsize=11)
axes[0].set_xlabel('Discount Band', color='#94a3b8')
axes[0].set_ylabel('Sales ($)', color='#94a3b8')

# Boxplot: Discount bin vs Profit
sns.boxplot(data=df, x='discount_bin', y='Profit', ax=axes[1],
            palette=PALETTE, showfliers=False)
axes[1].set_title('Profit Distribution by Discount Level', color='white', fontsize=11)
axes[1].set_xlabel('Discount Band', color='#94a3b8')
axes[1].set_ylabel('Profit ($)', color='#94a3b8')
axes[1].axhline(0, color='#ff6b6b', linewidth=1.2, linestyle='--', alpha=0.7)

# Bar: Avg sales by category and region
pivot = df.groupby(['Region','Category'])['Sales'].mean().unstack()
pivot.plot(kind='bar', ax=axes[2], color=PALETTE[:3], edgecolor='none', width=0.65)
axes[2].set_title('Avg Sales: Region × Category', color='white', fontsize=11)
axes[2].set_xlabel('Region', color='#94a3b8')
axes[2].set_ylabel('Avg Sales ($)', color='#94a3b8')
axes[2].legend(facecolor='#1a1d27', labelcolor='white', fontsize=8)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('../data/processed/eda_discount_analysis.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 5. Segment & Regional Summary Stats

In [ ]:
summary = df.groupby(['Segment','Category']).agg(
    Total_Sales=('Sales','sum'),
    Avg_Sales=('Sales','mean'),
    Avg_Profit=('Profit','mean'),
    Avg_Discount=('Discount','mean'),
    Order_Count=('Sales','count')
).round(2)

summary['Profit_Margin_%'] = (summary['Avg_Profit'] / summary['Avg_Sales'] * 100).round(1)
summary.style.background_gradient(cmap='Blues', subset=['Total_Sales']).format({
    'Total_Sales': '${:,.0f}',
    'Avg_Sales': '${:.2f}',
    'Avg_Profit': '${:.2f}',
    'Avg_Discount': '{:.1%}',
    'Profit_Margin_%': '{:.1f}%'
})

In [ ]:
print('\n=== EDA Complete ===')
print('Charts saved to data/processed/')
print('Proceed to: notebooks/02_model_prototyping.ipynb')